In [1]:
!pip install -q chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 71.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 76.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 65.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently t

In [2]:
import chromadb

# In-memory client (resets when runtime restarts) - great for learning
client = chromadb.Client()

# Create a fresh collection to practice CRUD operations on
collection = client.create_collection(name="genai_crud_demo")

print("Collection created:", collection.name)
print("Current document count:", collection.count())

Collection created: genai_crud_demo
Current document count: 0


In [3]:
collection.add(
    documents=[
        "Generative AI can create text, images, code, and other types of content.",
        "Large Language Models generate human-like text using learned patterns.",
        "Retrieval Augmented Generation retrieves relevant information before generating an answer.",
        "Document chunking divides large documents into smaller meaningful pieces.",
        "Vector databases store embeddings and help retrieve semantically similar information."
    ],
    metadatas=[
        {"category": "genai"},
        {"category": "llm"},
        {"category": "rag"},
        {"category": "rag"},
        {"category": "vector_db"}
    ],
    ids=["genai1", "genai2", "genai3", "genai4", "genai5"]
)

print("Documents added! Total count:", collection.count())

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:00<00:00, 95.3MiB/s]


Documents added! Total count: 5


In [4]:
preview = collection.peek(limit=3)

for doc_id, doc in zip(preview["ids"], preview["documents"]):
    print(doc_id, "->", doc)

genai1 -> Generative AI can create text, images, code, and other types of content.
genai2 -> Large Language Models generate human-like text using learned patterns.
genai3 -> Retrieval Augmented Generation retrieves relevant information before generating an answer.


In [5]:
result = collection.get(ids=["genai1", "genai3"])

for doc_id, doc, meta in zip(
    result["ids"],
    result["documents"],
    result["metadatas"]
):
    print(f"{doc_id} | {meta} | {doc}")

genai1 | {'category': 'genai'} | Generative AI can create text, images, code, and other types of content.
genai3 | {'category': 'rag'} | Retrieval Augmented Generation retrieves relevant information before generating an answer.


In [6]:
results = collection.query(
    query_texts=["How does retrieval augmented generation work?"],
    n_results=2
)

for doc_id, doc, distance in zip(
    results["ids"][0],
    results["documents"][0],
    results["distances"][0]
):
    print(f"{doc_id} | distance={distance:.4f} | {doc}")

genai3 | distance=0.5079 | Retrieval Augmented Generation retrieves relevant information before generating an answer.
genai1 | distance=1.4096 | Generative AI can create text, images, code, and other types of content.


In [7]:
results = collection.query(
    query_texts=["How are documents divided for RAG?"],
    n_results=3,
    where={"category": "rag"}
)

for doc_id, doc, distance in zip(
    results["ids"][0],
    results["documents"][0],
    results["distances"][0]
):
    print(f"{doc_id} | distance={distance:.4f} | {doc}")

genai4 | distance=0.8428 | Document chunking divides large documents into smaller meaningful pieces.
genai3 | distance=1.7318 | Retrieval Augmented Generation retrieves relevant information before generating an answer.


In [8]:
collection.update(
    ids=["genai4"],
    documents=[
        "Document chunking divides large documents into smaller meaningful pieces for efficient retrieval."
    ],
    metadatas=[
        {"category": "rag", "updated": True}
    ]
)

updated = collection.get(ids=["genai4"])

print(updated["documents"])
print(updated["metadatas"])

['Document chunking divides large documents into smaller meaningful pieces for efficient retrieval.']
[{'category': 'rag', 'updated': True}]


In [9]:
collection.upsert(
    documents=[
        "Vector databases store embeddings and efficiently retrieve semantically similar documents.",
        "Prompt engineering helps users design effective instructions for large language models."
    ],
    ids=["genai5", "genai6"]
)

print("Total count after upsert:", collection.count())
print(collection.get(ids=["genai5", "genai6"])["documents"])

Total count after upsert: 6
['Vector databases store embeddings and efficiently retrieve semantically similar documents.', 'Prompt engineering helps users design effective instructions for large language models.']


In [10]:
collection.delete(ids=["genai2"])

print("Total count after delete:", collection.count())
print("Remaining ids:", collection.get()["ids"])

Total count after delete: 5
Remaining ids: ['genai1', 'genai3', 'genai4', 'genai5', 'genai6']


In [11]:
collection.delete(where={"category": "genai"})

print("Total count after category delete:", collection.count())
print("Remaining ids:", collection.get()["ids"])

Total count after category delete: 4
Remaining ids: ['genai3', 'genai4', 'genai5', 'genai6']


In [12]:
final_state = collection.get()

print("Final document count:", collection.count())

for doc_id, doc, meta in zip(
    final_state["ids"],
    final_state["documents"],
    final_state["metadatas"]
):
    print(f"{doc_id} | {meta} | {doc}")

Final document count: 4
genai3 | {'category': 'rag'} | Retrieval Augmented Generation retrieves relevant information before generating an answer.
genai4 | {'category': 'rag', 'updated': True} | Document chunking divides large documents into smaller meaningful pieces for efficient retrieval.
genai5 | {'category': 'vector_db'} | Vector databases store embeddings and efficiently retrieve semantically similar documents.
genai6 | None | Prompt engineering helps users design effective instructions for large language models.
